In [ ]:
import torch
import json
import requests
import yaml
from openai import OpenAI
from transformers import GPT2LMHeadModel, PreTrainedTokenizerFast
from transformers import AutoModelForCausalLM, AutoTokenizer, GPTNeoForCausalLM

# ============ LOAD YOUR MODEL ============
your_model_path = "./runs/finalised_best_model/final_model"
your_model = GPT2LMHeadModel.from_pretrained(your_model_path)
your_tok   = PreTrainedTokenizerFast.from_pretrained(your_model_path)
your_model.eval()
print("Your model loaded ✅")

# ============ LOAD REPLICATED GPT-NEO 33M ============
neo_model_path = "./runs/gptneo_33M_replicated/final_model"
neo_model = GPTNeoForCausalLM.from_pretrained(neo_model_path)
neo_tok   = PreTrainedTokenizerFast.from_pretrained(neo_model_path)
neo_model.eval()
print("Replicated GPT-Neo 33M loaded ✅") 

# ============ OPENAI CLIENT ============
client = OpenAI(
    api_key  = "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",   # replace with your xAI API key
    base_url = "https://api.groq.com/openai/v1", # Grok uses OpenAI-compatible API
)

# ============ FETCH OFFICIAL EVALUATION PROMPTS ============
def fetch_official_prompts():
    url = "https://huggingface.co/datasets/roneneldan/TinyStories/resolve/main/Evaluation%20prompts.yaml"
    try:
        response = requests.get(url)
        response.raise_for_status()
        prompts_data = yaml.safe_load(response.text)

        prompts = []
        for item in prompts_data:
            if isinstance(item, dict):
                prompt = item.get("prompt") or item.get("Prompt") or list(item.values())[0]
                prompts.append(prompt)
            elif isinstance(item, str):
                prompts.append(item)

        print(f"Loaded {len(prompts)} official evaluation prompts ✅")
        return prompts

    except Exception as e:
        print(f"Could not fetch official prompts: {e}")
        print("Falling back to manual prompts...")
        return [
            "Once upon a time there was a little girl",
            "One day a puppy named Max went to the park",
            "The princess lived in a big castle",
            "Tom was very sad because he lost his toy",
            "In the forest there lived a friendly dragon",
        ]

prompts = fetch_official_prompts()
prompts = prompts[:50]
print(f"Using {len(prompts)} prompts for evaluation")

# ============ GENERATION FUNCTION ============
def generate_story(model, tokenizer, prompt, max_new_tokens=200):
    inputs = tokenizer(prompt, return_tensors="pt")
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=1.0,
            top_p=0.95,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id,
        )
    full_text = tokenizer.decode(output[0], skip_special_tokens=True)
    generated = full_text[len(prompt):].strip()
    return generated

# ============ GPT EVAL FUNCTION ============
def gpt_eval(prompt, story, model_name="openai/gpt-oss-20b"):
    eval_prompt = f"""the following exercise, the student is given a beginning of a story. The student needs to complete it into a full story.
    The exercise tests the student´s language abilities and creativity. 
    The symbol *** marks the separator between the prescribed beginning and the student’s completion: {prompt}***{story}

    Please provide your general assessment about the part written by the student (the one after the *** symbol). Is it gramatically correct? Is it consistent with the beginning of the story? 
    Pay special attention to whether the student manages to complete the sentence which is split in the middle by the separator ***. 

    Now, grade the student’s completion in terms of grammar (1-10), creativity (1-10), consistency (1-10), plot (1-10), 
    with the story’s beginning and whether the plot makes sense. 


Return ONLY valid JSON matching this schema:
{{
  "grammar": 1,
  "creativity": 1,
  "consistency": 1,
  "plot": 1
}}"""

    response = client.responses.create(
        model=model_name,
        input=eval_prompt,
        text={
            "format": {
                "type": "json_schema",
                "name": "story_eval",
                "schema": {
                    "type": "object",
                    "properties": {
                        "grammar":      {"type": "integer", "minimum": 1, "maximum": 10},
                        "creativity":   {"type": "integer", "minimum": 1, "maximum": 10},
                        "consistency":  {"type": "integer", "minimum": 1, "maximum": 10},
                        "plot":         {"type": "integer", "minimum": 1, "maximum": 10},
                    },
                    "required": ["grammar", "creativity", "consistency", "plot"],
                    "additionalProperties": False
                },
                "strict": True
            }
        }
    )

    scores = json.loads(response.output_text)
    scores["overall"] = round(
        (scores["grammar"] + scores["creativity"] + scores["consistency"] + scores["plot"]) / 3, 2
    )
    return scores

# ============ EVALUATE ONE MODEL ============
def evaluate_model(model, tokenizer, model_label, prompts, n_completions=10, judge_model="openai/gpt-oss-20b"):
    print(f"\n{'='*60}")
    print(f"Evaluating {model_label}...")
    print(f"{'='*60}")

    all_grammar     = []
    all_creativity  = []
    all_consistency = []
    all_plot = []
    all_overall     = []
    results         = []

    for i, prompt in enumerate(prompts):
        print(f"\nPrompt {i+1}/{len(prompts)}: '{prompt}'")

        prompt_grammar     = []
        prompt_creativity  = []
        prompt_consistency = []
        prompt_plot = []
        prompt_stories = []

        for j in range(n_completions):
            story  = generate_story(model, tokenizer, prompt)
            scores = gpt_eval(prompt, story, model_name=judge_model)

            prompt_grammar.append(scores["grammar"])
            prompt_creativity.append(scores["creativity"])
            prompt_consistency.append(scores["consistency"])
            prompt_plot.append(scores["plot"])
            prompt_stories.append(story)

            print(
                f"  [{j+1}/{n_completions}] "
                f"grammar={scores['grammar']} "
                f"creativity={scores['creativity']} "
                f"consistency={scores['consistency']} "
                f"plot={scores['plot']} "
                f"overall={scores['overall']}"
            )

        avg_grammar     = round(sum(prompt_grammar)     / n_completions, 2)
        avg_creativity  = round(sum(prompt_creativity)  / n_completions, 2)
        avg_consistency = round(sum(prompt_consistency) / n_completions, 2)
        avg_plot = round(sum(prompt_plot) / n_completions, 2)
        avg_overall     = round((avg_grammar + avg_creativity + avg_consistency + avg_plot) / 4, 2)

        all_grammar.append(avg_grammar)
        all_creativity.append(avg_creativity)
        all_consistency.append(avg_consistency)
        all_plot.append(avg_plot)
        all_overall.append(avg_overall)

        results.append({
            "prompt"          : prompt,
            "stories"         : prompt_stories,
            "avg_grammar"     : avg_grammar,
            "avg_creativity"  : avg_creativity,
            "avg_consistency" : avg_consistency,
            "avg_plot"        : avg_plot,
            "avg_overall"     : avg_overall,
        })

        print(
            f"  → Prompt average: "
            f"grammar={avg_grammar} "
            f"creativity={avg_creativity} "
            f"consistency={avg_consistency} "
            f"plot={avg_plot} "
            f"overall={avg_overall}"
        )

    final = {
        "model"       : model_label,
        "grammar"     : round(sum(all_grammar)     / len(prompts), 2),
        "creativity"  : round(sum(all_creativity)  / len(prompts), 2),
        "consistency" : round(sum(all_consistency) / len(prompts), 2),
        "plot"        : round(sum(all_plot)        / len(prompts), 2),
        "overall"     : round(sum(all_overall)     / len(prompts), 2),
        "per_prompt"  : results,
    }

    return final

# ============ RUN EVALUATION ============
N_COMPLETIONS = 10
JUDGE_MODEL   = "openai/gpt-oss-20b"

your_results = evaluate_model(
    your_model, your_tok,
    "Your Model (91.5M GPT-2, ablation params)",
    prompts,
    n_completions=N_COMPLETIONS,
    judge_model=JUDGE_MODEL
)

neo_results = evaluate_model(
    neo_model, neo_tok,
    "Replicated GPT-Neo 33M (standard params, 1 epoch)",
    prompts,
    n_completions=N_COMPLETIONS,
    judge_model=JUDGE_MODEL
)

# ============ SAVE RESULTS ============
all_results = {
    "evaluation_config": {
        "n_prompts"     : len(prompts),
        "n_completions" : N_COMPLETIONS,
        "total_stories" : len(prompts) * N_COMPLETIONS * 2,
        "evaluator"     : JUDGE_MODEL,
        "methodology"   : "LLM-as-a-judge replicating TinyStories GPT-eval framework",
        "comparison"    : "Equal training budget (1 epoch each)"
    },
    "your_model" : your_results,
    "neo_33m"    : neo_results,
}

with open("./gpt_eval_comparison1.json", "w") as f:
    json.dump(all_results, f, indent=2)

print("\nResults saved to gpt_eval_comparison1.json ✅")

# ============ PRINT FINAL COMPARISON ============
print(f"\n{'='*65}")
print("COMPARISON 1 — EQUAL TRAINING BUDGET")
print("Your 91.7M (ablation) vs Replicated GPT-Neo 33M (standard)")
print(f"{len(prompts)} prompts × {N_COMPLETIONS} completions = "
      f"{len(prompts)*N_COMPLETIONS} stories per model")
print(f"{'='*65}")

print(f"\n{'Metric':<15} {'Your 91.7M':>15} {'GPT-Neo 33M':>15} {'Winner':>10}")
print(f"{'─'*57}")

your_wins = 0
neo_wins  = 0

for metric in ["grammar", "creativity", "consistency", "plot", "overall"]:
    your_val = your_results[metric]
    neo_val  = neo_results[metric]

    if your_val >= neo_val:
        winner = "YOURS ✅"
        your_wins += 1
    else:
        winner = "NEO ✅"
        neo_wins += 1

    print(f"{metric:<15} {str(your_val):>15} {str(neo_val):>15} {winner:>10}")

print(f"{'─'*57}")
print(f"{'Total wins':<15} {str(your_wins):>15} {str(neo_wins):>15}")
print(f"\n{'='*65}")
print(f"Context:")
print(f"  Your model:    91.7M params, GPT-2 arch, ablation hyperparams")
print(f"  GPT-Neo 33M:   31.1M params, GPT-Neo arch, standard hyperparams")
print(f"  Both trained:  1 epoch on TinyStories")
print(f"{'='*65}")